# CGPA Prediction & Academic Optimization Model

**NIHUB · IT & Innovation — AI/Machine Learning Track, FUT Minna**

This notebook builds a supervised regression model that predicts a student's current CGPA from prior academic performance and behavioural survey data, compares two algorithms, and produces actionable, feature-importance-driven recommendations.

**Target variable:** `Current CGPA` (the student's cumulative CGPA as of their most recently completed semester).

**Data:** anonymous Google Form responses from fellow students — no names, emails, or matric numbers were collected.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import re
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


## 2. Data Loading

Load the raw Google Forms export, keep only the 13 columns collected on the form (dropping the auto-added Timestamp), and give each column a clean, readable name.

In [ ]:
df = pd.read_csv("./Dataset/Student_dataset.csv")


In [ ]:
df = df.iloc[:, :13].copy()


In [ ]:
df.columns = ['Timestamp', 'Current CGPA', 'Previous CGPA', 
              'Units taken', 'Average CA score', 'No of higher unit course', 
              'Test preparedness', 'Assignment commitment', 'Procrastination tendency', 
              'Weekly Study hours', 'Atendance rate', 
              'Extracurricular hours', 'Access to PQ' ]
df = df.drop(columns=['Timestamp'], errors='ignore')
df.head()


In [ ]:
df.info()


## 3. Cleaning the Target & Previous CGPA

These two columns are cleaned first, before anything else, because every later row-drop decision depends on having a trustworthy target. Free-text entries like *"Not sure, around 4.32"* are parsed for the embedded number; anything outside the valid 0.0–5.0 CGPA range is treated as invalid and set to missing.

In [ ]:
def clean_cgpa(val):
    if pd.isna(val): return np.nan
    val_str = str(val).strip()
    if 'Not sure' in val_str:
        m = re.search(r'(\d+\.\d+)', val_str)
        return float(m.group(1)) if m else np.nan
    try:
        res = float(val_str)
        return res if 0.0 <= res <= 5.0 else np.nan
    except ValueError:
        return np.nan

df['Current CGPA'] = df['Current CGPA'].apply(clean_cgpa)
df['Previous CGPA'] = df['Previous CGPA'].apply(clean_cgpa)


In [ ]:
# A missing target or missing Previous CGPA makes a row unusable for training
df = df.dropna(subset=['Current CGPA', 'Previous CGPA']).reset_index(drop=True)


In [ ]:
df.info()


## 4. First Look at the Target

A quick check on the one column we already trust, before touching anything else.

In [ ]:
sns.histplot(data=df, x='Current CGPA', bins=10, kde=True)
plt.title('Distribution of Current CGPA')
plt.show()


In [ ]:
plt.figure(figsize=(5,5))
sns.scatterplot(data=df, x='Previous CGPA', y='Current CGPA')
plt.plot([2,5],[2,5],'r--', label='no change')
plt.legend(); plt.title('Previous vs Current CGPA'); plt.show()


## 5. Cleaning the Remaining Features

The rest of the columns contain a lot of messy free-text (e.g. *"104 units"*, *"Barely an hr sometimes"*, *"Dont know"*), so each one gets its own cleaning function, with values outside a plausible range or unparseable text converted to missing rather than guessed at.

- **Attendance rate**: some respondents answered on the intended 1–5 scale, others answered as a raw percentage — percentages are converted onto the 1–5 scale.
- **Units taken**: numeric value extracted from free text, kept only within a plausible 10–24 range.
- **Average CA score**: kept within the valid 0–40 mark range.
- **Weekly Study hours / Extracurricular hours**: numeric value extracted from free text; recognisable non-numeric answers (*"can't estimate"*) become missing. Every genuinely entered number, however small, is kept as-is — no values are dropped just for being low.
- **No of higher-unit course**: kept within a plausible 0–10 range.
- **Access to PQ**: standardised to Yes/No, then mapped to 1.0/0.0.

In [ ]:
def clean_attendance(v):
    if pd.isna(v):
        return np.nan
    if v > 5:                                   # old form: percentage
        return float(np.clip(np.ceil(v / 20), 1, 5))
    return v

df['Atendance rate'] = df['Atendance rate'].apply(clean_attendance)


def clean_units_series(series):
    cleaned = []
    for val in series:
        if pd.isna(val):
            cleaned.append(np.nan)
            continue
        val_str = str(val).strip().lower()
        nums = re.findall(r'\d+\.?\d*', val_str)
        if nums:
            num = float(nums[0])
            cleaned.append(num if 10 <= num <= 24 else np.nan)
        else:
            cleaned.append(np.nan)
    return cleaned


def clean_ca(val):
    if pd.isna(val): return np.nan
    try:
        num = float(str(val).strip())
        return num if 0 <= num <= 40 else np.nan
    except ValueError:
        return np.nan

df['Average CA score'] = df['Average CA score'].apply(clean_ca)


def clean_hours_series(series, cap=112, ambiguous_low=False):
    cleaned = []
    for val in series:
        if pd.isna(val):
            cleaned.append(np.nan); continue
        s = str(val).strip().lower()

        if s in ['nil', 'nill', 'none', 'non', '0', '0.0']:
            cleaned.append(0.0); continue

        if any(x in s for x in ["can't estimate", "the whole week", "sport", "i can't", "barely"]):
            cleaned.append(np.nan); continue

        m = re.search(r'(\d+\.?\d*)\s*(?:-|to)\s*(\d+\.?\d*)', s)
        if m:
            num = (float(m.group(1)) + float(m.group(2))) / 2
        else:
            nums = re.findall(r'\d+\.?\d*', s)
            if not nums:
                cleaned.append(np.nan); continue
            num = float(nums[0])

        if num > cap:
            cleaned.append(np.nan)
        elif ambiguous_low and 0 < num <= 3:
            cleaned.append(np.nan)
        else:
            cleaned.append(num)
    return cleaned


for col in df.columns:
    if col == 'Units taken':
        df[col] = clean_units_series(df[col])
    elif col == 'Weekly Study hours':
        df[col] = clean_hours_series(df[col], cap=112)
    elif col == 'Extracurricular hours':
        df[col] = clean_hours_series(df[col], cap=40)


def clean_higher_units(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().lower()
    try:
        num = float(val_str)
        return num if 0 <= num <= 10 else np.nan
    except ValueError:
        return np.nan

df['No of higher unit course'] = df['No of higher unit course'].apply(clean_higher_units)


def clean_pq(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().lower()
    return 'Yes' if 'yes' in val_str else 'No'

df['Access to PQ'] = df['Access to PQ'].apply(clean_pq)
df['Access to PQ'] = df['Access to PQ'].map({'Yes': 1.0, 'No': 0.0})


## 6. Cross-Imputing Units Taken ↔ No of Higher-Unit Courses

These two fields are logically related (a student missing one often still gave the other), so each is used to fill gaps in the other via the most common matching value, before falling back to a general median later.

In [ ]:
def get_clean_mode(series):
    valid_series = series.dropna()
    if len(valid_series) > 0:
        return valid_series.mode().iloc[0]
    return np.nan

units_mode = df.groupby('No of higher unit course')['Units taken'].transform(get_clean_mode)
df['Units taken'] = df['Units taken'].fillna(units_mode)

higher_units_mode = df.groupby('Units taken')['No of higher unit course'].transform(get_clean_mode)
df['No of higher unit course'] = df['No of higher unit course'].fillna(higher_units_mode)


In [ ]:
df.info()


## 7. Exploratory Data Analysis (on cleaned data)

With every column now numeric, we can look at distributions, outliers, and relationships with the target without the raw free-text entries breaking the plots.

In [ ]:
cols = df.columns.drop('Current CGPA')
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, c in zip(axes.ravel(), cols):
    sns.boxplot(y=df[c], ax=ax); ax.set_title(c, fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, c in zip(axes.ravel(), cols):
    sns.scatterplot(data=df, x=c, y='Current CGPA', ax=ax)
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()


**Reading the correlation matrix:**

- `Previous CGPA` is, unsurprisingly, the strongest correlate of `Current CGPA` (0.95) — expected, since cumulative CGPA carries forward from one semester to the next.
- `Attendance rate` (0.51) and `Average CA score` (0.45) are the next strongest, and both are genuinely actionable behaviours.
- `Weekly Study hours` is weakly **negatively** correlated with CGPA (−0.24). This is most likely reverse causation in a small, self-reported sample — students who are already struggling tend to report studying more (compensating/cramming), not the other way round. Given the small sample size and weak correlation, this feature is **not** used as a positive recommendation driver below.

## 8. Handling Remaining Missing Values

Rows still missing `Units taken` after cross-imputation are dropped (too little information to recover). Remaining gaps in the behavioural columns are filled with the column median, a reasonable choice for small, roughly-symmetric survey scales.

In [ ]:
df = df.dropna(subset=['Units taken']).reset_index(drop=True)

behavioral_cols = [
    'No of higher unit course', 'Assignment commitment', 'Procrastination tendency',
    'Weekly Study hours', 'Average CA score', 'Atendance rate',
    'Extracurricular hours', 'Access to PQ'
]

for col in behavioral_cols:
    df[col] = df[col].fillna(df[col].median())

print(df.isna().sum())


In [ ]:
df.info()


## 9. Feature Set

`X` is every column except the target; `y` is `Current CGPA`.

In [ ]:
X = df.drop(columns=['Current CGPA'])
y = df['Current CGPA']

print(X.shape, y.shape)


## 10. Model Building

Given only ~37 rows, a single train/test split would leave an unreliably small test set, so **5-fold cross-validation** is used instead of a held-out split. Each model is wrapped in a `Pipeline` (median imputation → scaling → model) so these steps are refit fresh inside every fold, avoiding any leakage between folds.

Three models are compared:
- **Mean baseline** (`DummyRegressor`) — always predicts the average CGPA. The floor every real model must beat.
- **Ridge Regression** (`RidgeCV`, searching over several regularisation strengths) — simple and directly interpretable via its coefficients.
- **Random Forest Regressor** — an ensemble of decision trees, able to capture non-linear patterns the linear model can't, at some cost to direct interpretability.

In [ ]:
def make(model):
    return Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', model),
    ])

models = {
    'Mean baseline': make(DummyRegressor()),
    'Ridge':         make(RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0])),
    'RandomForest':  make(RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42)),
}


## 11. K-Fold Cross-Validation Comparison

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'mae': 'neg_mean_absolute_error', 'mse': 'neg_mean_squared_error', 'r2': 'r2'}

for name, pipe in models.items():
    s = cross_validate(pipe, X, y, cv=kf, scoring=scoring)
    mae, mse, r2 = -s['test_mae'], -s['test_mse'], s['test_r2']
    print(f"{name:14s} MAE {mae.mean():.3f}±{mae.std():.3f} | RMSE {np.sqrt(mse).mean():.3f} | R² {r2.mean():.3f}±{r2.std():.3f}")


**Results summary:**

| Model | MAE | RMSE | R² |
|---|---|---|---|
| Mean baseline | 0.677 | 0.808 | −0.468 |
| Ridge | 0.232 | 0.299 | 0.774 |
| Random Forest | 0.197 | 0.270 | 0.853 |

Both real models clearly beat the baseline, confirming the features carry genuine signal. **Random Forest edges out Ridge on accuracy**, but its MAE also varies a little more across folds — expected with only 37 rows. **Ridge stays the more directly interpretable model.** This is the comparison the brief asks for: one accurate, flexible model vs. one simpler, more transparent one, each with a real trade-off rather than a flat win.

## 12. Feature Importance

Both models are refit on the full dataset (since we used cross-validation rather than a held-out split, all available data is used for the final model).

In [ ]:
ridge_pipe = models['Ridge']
ridge_pipe.fit(X, y)

rf_pipe = models['RandomForest']
rf_pipe.fit(X, y)


In [ ]:
ridge_coefs = pd.Series(ridge_pipe.named_steps['model'].coef_, index=X.columns)
ridge_coefs.sort_values(key=abs, ascending=False)


In [ ]:
rf_importances = pd.Series(rf_pipe.named_steps['model'].feature_importances_, index=X.columns)
rf_importances.sort_values(ascending=False)


**Interpreting feature importance:**

- `Previous CGPA` dominates both rankings — expected, since `Current CGPA` is partly derived from it. It is **excluded** from the recommendations below, since "have a better previous CGPA" isn't actionable advice.
- Excluding Previous CGPA, the most useful, actionable features — combining each feature's raw correlation direction (from the heatmap) with its Random Forest importance — are: **Attendance rate, Average CA score, Assignment commitment**, and **Procrastination tendency**.
- `Weekly Study hours` shows an unstable/negative sign under Ridge due to multicollinearity with other behavioural features in this small sample (see Section 7) — it is deliberately left out of the recommendation logic below rather than risk giving misleading advice.

## 13. Optimization Layer — Recommendations

A simple, transparent function compares a student's values against the dataset's median for each actionable feature, and returns plain-language suggestions for whichever ones fall on the unfavourable side. This logic is the same one used in the Streamlit app below.

In [ ]:
feature_medians = X.median()

def generate_recommendations(input_dict, medians=feature_medians):
    tips = []
    if input_dict['Atendance rate'] < medians['Atendance rate']:
        tips.append("Your attendance is below the typical level in this dataset — attendance "
                    "had the strongest link to CGPA among behavioural factors, so improving it "
                    "could meaningfully help.")
    if input_dict['Procrastination tendency'] > medians['Procrastination tendency']:
        tips.append("Your procrastination rating is higher than typical — lower procrastination "
                    "was linked to better outcomes.")
    if input_dict['Assignment commitment'] < medians['Assignment commitment']:
        tips.append("Your assignment commitment is below typical — staying consistent with "
                    "assignments was linked to better CGPA.")
    if input_dict['Average CA score'] < medians['Average CA score']:
        tips.append("Your CA score is below typical — since CA feeds directly into your overall "
                    "result, focusing here could help.")
    if not tips:
        tips.append("Your behavioural indicators are already at or above typical levels in this "
                    "dataset — keep it up!")
    return tips

# Example: test on the first student in the dataset
example_input = X.iloc[0].to_dict()
predicted = rf_pipe.predict(X.iloc[[0]])[0]
print(f"Predicted CGPA: {predicted:.2f}")
for tip in generate_recommendations(example_input):
    print('-', tip)


## 14. Saving the Model for Streamlit Deployment

The Random Forest pipeline is saved as the deployed model (best cross-validated accuracy), along with the feature medians the recommendation logic depends on. `app.py` (in the same folder) loads these two files and serves the prediction + recommendation tool.

In [ ]:
joblib.dump(rf_pipe, 'rf_model.pkl')
joblib.dump(feature_medians, 'feature_medians.pkl')
print('Saved rf_model.pkl and feature_medians.pkl')
